In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#installing libraries for BERT
!pip install datasets
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.3 MB/s eta 0:00:00


In [ ]:
## From lab 7
## This code here just makes it so you don't need an API
## key for Weights and Biases. Just run it, and you're good.

import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from datasets import load_dataset

csv_file_path = '/content/drive/MyDrive/2025-26/NLP/dancing1000v2.csv'

#load the dataset from the local CSV file
dancing_dataset = load_dataset('csv', data_files=csv_file_path)

#display the first few examples of the dataset
print(dancing_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres'],
        num_rows: 1000
    })
})


In [ ]:
#create a randomized 80/20 train-test split
train_test_dataset = dancing_dataset['train'].train_test_split(test_size=0.2, seed=42)

print(train_test_dataset)

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres'],
        num_rows: 800
    })
    test: Dataset({
        features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres'],
        num_rows: 200
    })
})


In [ ]:
#from lab 7
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
#define a tokenization function
def tokenize_function(examples):
    return tokenizer(examples['lyrics'], truncation=True, padding=True)

In [ ]:
# from lab 7
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
#function to create labels based on danceability score
def create_danceability_labels(example):
    danceability = example['danceability']
    if danceability < 0.4:
        example['labels'] = 0 # 'low danceability'
    elif danceability > 0.6:
        example['labels'] = 1 # 'high danceability'
    else:
        example['labels'] = -1 # Mark for removal or as 'intermediate'
    return example

#apply the labeling function to both train and test splits
train_test_dataset_labeled = train_test_dataset.map(create_danceability_labels)

#filter out examples that are not 'low danceability' or 'high danceability'
train_test_dataset_filtered = train_test_dataset_labeled.filter(lambda example: example['labels'] != -1)

#update the 'train' and 'test' splits with the filtered data
train_dataset = train_test_dataset_filtered['train']
test_dataset = train_test_dataset_filtered['test']

#display the updated datasets to show the new 'labels' column and filtered counts
print("Filtered Training Data:", train_dataset)
print("Filtered Test Data:", test_dataset)

#apply tokenization to the filtered datasets
tokenized_train_lyrics = train_dataset.map(tokenize_function, batched=True)
tokenized_test_lyrics = test_dataset.map(tokenize_function, batched=True)

print("Tokenized and Filtered Training Data:", tokenized_train_lyrics)
print("Tokenized and Filtered Test Data:", tokenized_test_lyrics)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

Filtered Training Data: Dataset({
    features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres', 'labels'],
    num_rows: 800
})
Filtered Test Data: Dataset({
    features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres', 'labels'],
    num_rows: 200
})


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenized and Filtered Training Data: Dataset({
    features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 800
})
Tokenized and Filtered Test Data: Dataset({
    features: ['Unnamed: 0', 'id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 200
})


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from evaluate import load
import numpy as np

# Load the model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Define the evaluation metric
metric = load("accuracy")

def compute_metrics(eval_pred):
   load_accuracy = load("accuracy")
   load_f1 = load("f1")

   logits, labels = eval_pred
   predictions = np.argmax(logits, axis=-1)
   accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
   f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
   return {"accuracy": accuracy, "f1": f1}

print("Model loaded:", model.__class__.__name__)
print("Metric loaded:", metric.__class__.__name__)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: DistilBertForSequenceClassification
Metric loaded: Accuracy


In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/2025-26/NLP/results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_lyrics,
    eval_dataset=tokenized_test_lyrics,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.4288141174316406, metrics={'train_runtime': 263.7561, 'train_samples_per_second': 15.166, 'train_steps_per_second': 0.948, 'total_flos': 529869594624000.0, 'train_loss': 0.4288141174316406, 'epoch': 5.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.437109112739563,
 'eval_accuracy': 0.81,
 'eval_f1': 0.7956989247311828,
 'eval_runtime': 6.5263,
 'eval_samples_per_second': 30.645,
 'eval_steps_per_second': 1.992,
 'epoch': 5.0}

In [ ]:
#printing a few example predictions

predictions = trainer.predict(tokenized_test_lyrics)
predicted_labels = np.argmax(predictions.predictions, axis=1)

def label_to_text(label):
    if label == 0:
        return 'low danceability'
    elif label == 1:
        return 'high danceability'
    else:
        return 'unknown'

print("\n--- Sample Predictions vs. Actual Labels ---")
for i in range(10):
    lyrics = tokenized_test_lyrics[i]['lyrics']
    song_name = tokenized_test_lyrics[i]['name']
    artist_name = tokenized_test_lyrics[i]['artists']
    numerical_danceability = tokenized_test_lyrics[i]['danceability']
    actual_label = label_to_text(tokenized_test_lyrics[i]['labels'])
    predicted_label = label_to_text(predicted_labels[i])

    print(f"\nExample {i+1}:")
    print(f"Song Title: {song_name}")
    print(f"Artist: {artist_name}")
    print(f"Numerical Danceability: {numerical_danceability:.2f}")
    print(f"Actual Danceability: {actual_label}")
    print(f"Predicted Danceability: {predicted_label}")



--- Sample Predictions vs. Actual Labels ---

Example 1:
Song Title: Hawk as Weapon
Artist: ["Conan"]
Numerical Danceability: 0.34
Actual Danceability: low danceability
Predicted Danceability: low danceability

Example 2:
Song Title: My Funeral
Artist: ["Dark Funeral"]
Numerical Danceability: 0.15
Actual Danceability: low danceability
Predicted Danceability: low danceability

Example 3:
Song Title: Reminder
Artist: ["Moderat"]
Numerical Danceability: 0.40
Actual Danceability: low danceability
Predicted Danceability: low danceability

Example 4:
Song Title: Hark! The Herald Angels Sing
Artist: ["Susan Boyle"]
Numerical Danceability: 0.25
Actual Danceability: low danceability
Predicted Danceability: low danceability

Example 5:
Song Title: Soothe Me
Artist: ["Sam & Dave"]
Numerical Danceability: 0.75
Actual Danceability: high danceability
Predicted Danceability: high danceability

Example 6:
Song Title: Tattooed Dancer
Artist: ["Ozzy Osbourne"]
Numerical Danceability: 0.26
Actual Dancea

In [ ]:
from scipy.special import softmax

logits = predictions.predictions

probs = softmax(logits, axis=1)

high_class_probs = probs[:, 1]

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

predictions = trainer.predict(tokenized_test_lyrics)
logits = predictions.predictions
predicted_labels = np.argmax(logits, axis=1)

true_labels = np.array(tokenized_test_lyrics["labels"])

accuracy = accuracy_score(true_labels, predicted_labels)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average="binary")

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=["low danceability", "high danceability"]))

Accuracy: 0.8100
Precision: 0.8222
Recall: 0.7708
F1 Score: 0.7957

Classification Report:
                   precision    recall  f1-score   support

 low danceability       0.80      0.85      0.82       104
high danceability       0.82      0.77      0.80        96

         accuracy                           0.81       200
        macro avg       0.81      0.81      0.81       200
     weighted avg       0.81      0.81      0.81       200



In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

true_danceability = np.array(tokenized_test_lyrics["danceability"])
predicted_score = high_class_probs

mae = mean_absolute_error(true_danceability, predicted_score)
mse = mean_squared_error(true_danceability, predicted_score)
rmse = np.sqrt(mse)

print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")

MAE: 0.2288
MSE: 0.0752
RMSE: 0.2743


In [ ]:
from scipy.stats import spearmanr

true_danceability = np.array(tokenized_test_lyrics["danceability"])

spearman_corr, spearman_p = spearmanr(true_danceability, high_class_probs)

print(f"Spearman Correlation: {spearman_corr:.4f}")
print(f"Spearman p-value: {spearman_p:.4f}")

Spearman Correlation: 0.6769
Spearman p-value: 0.0000


In [ ]:
sns.set_style("whitegrid")

save_dir = "/content/drive/MyDrive/2025-26/NLP/plots/"
os.makedirs(save_dir, exist_ok=True)

metrics_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Value': [accuracy, precision, recall, f1]
}
df_metrics = pd.DataFrame(metrics_data)

plt.figure(figsize=(8, 5))
sns.barplot(x='Metric', y='Value', data=df_metrics, palette='viridis')
plt.title('Model Classification Performance Metrics', fontsize=14)
plt.ylim(0, 1)
plt.ylabel('Score', fontsize=12)
plt.xlabel('')
plt.xticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'classification_metrics_barplot.pdf'))
plt.close()

cm = confusion_matrix(true_labels, predicted_labels)
labels = ['low danceability', 'high danceability']

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels,
            cbar_kws={'label': 'Number of Samples'})
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10, rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'confusion_matrix_heatmap.pdf'))
plt.close()

reg_metrics_data = {
    'Metric': ['MAE', 'MSE', 'RMSE'],
    'Value': [mae, mse, rmse]
}
df_reg_metrics = pd.DataFrame(reg_metrics_data)

plt.figure(figsize=(8, 5))
sns.barplot(x='Metric', y='Value', data=df_reg_metrics, palette='mako')
plt.title('Model Regression Performance Metrics (Danceability Prediction)', fontsize=14)
plt.ylabel('Error Value', fontsize=12)
plt.xlabel('')
plt.yticks(fontsize=10)
plt.xticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'regression_metrics_barplot.pdf'))
plt.close()

plt.figure(figsize=(10, 7))
sns.scatterplot(x=true_danceability, y=high_class_probs, alpha=0.6, s=70, edgecolor='w', linewidth=0.5,
                color='purple')
plt.title(f'True Danceability vs. Predicted High Danceability Probability\n(Spearman Correlation: {spearman_corr:.4f})', fontsize=14)
plt.xlabel('True Danceability Score', fontsize=12)
plt.ylabel('Predicted Probability of High Danceability', fontsize=12)
plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 1.05)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.7)
plt.axhline(0.5, color='red', linestyle='--', label='Classification Threshold (0.5)', alpha=0.8)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'danceability_scatter_plot.pdf'))
plt.close()

/tmp/ipykernel_2756/4269110831.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Metric', y='Value', data=df_metrics, palette='viridis')
/tmp/ipykernel_2756/4269110831.py:53: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Metric', y='Value', data=df_reg_metrics, palette='mako')


In [ ]:
save_dir = "/content/drive/MyDrive/2025-26/NLP/plots/"

df_results = pd.DataFrame({
    'true_danceability': true_danceability,
    'true_labels': true_labels,
    'predicted_labels': predicted_labels
})

bins = np.arange(0, 1.1, 0.1)
labels = [f'{i:.1f}-{i+0.1:.1f}' for i in bins[:-1]]

df_results['danceability_bin'] = pd.cut(df_results['true_danceability'], bins=bins, labels=labels, right=False)

bin_accuracy = df_results.groupby('danceability_bin').apply(
    lambda x: accuracy_score(x['true_labels'], x['predicted_labels']) if len(x) > 0 else np.nan
).reset_index(name='model_accuracy')

bin_counts = df_results['danceability_bin'].value_counts().sort_index().reset_index(name='sample_count')
bin_counts.columns = ['danceability_bin', 'sample_count']

bin_performance = pd.merge(bin_accuracy, bin_counts, on='danceability_bin', how='left')

bin_performance['model_accuracy'] = bin_performance['model_accuracy'].fillna(np.nan)

bin_performance['random_baseline_accuracy'] = 0.5

plt.figure(figsize=(14, 8))

sns.lineplot(x='danceability_bin', y='model_accuracy', data=bin_performance, marker='o', label='Model Accuracy', color='blue', linewidth=2)
sns.lineplot(x='danceability_bin', y='random_baseline_accuracy', data=bin_performance, linestyle='--', color='red', label='Random Baseline (50%)', linewidth=2)

plt.title('Model Classification Accuracy vs. Random Baseline Across True Danceability Bins', fontsize=16)
plt.xlabel('True Danceability Range', fontsize=14)
plt.ylabel('Classification Accuracy', fontsize=14)
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'accuracy_by_danceability_bin_lineplot.pdf'))
plt.close()

plt.figure(figsize=(14, 5))
sns.barplot(x='danceability_bin', y='sample_count', data=bin_performance, palette='viridis')
plt.title('Number of Samples in Each True Danceability Bin', fontsize=16)
plt.xlabel('True Danceability Range', fontsize=14)
plt.ylabel('Number of Samples', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'samples_per_danceability_bin_barplot.pdf'))
plt.close()

/tmp/ipykernel_2756/2203386321.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bin_accuracy = df_results.groupby('danceability_bin').apply(
/tmp/ipykernel_2756/2203386321.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bin_accuracy = df_results.groupby('danceability_bin').apply(
/tmp/ipykernel_2756/2203386321.py:55: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  

In [ ]:
display(bin_performance)

,danceability_bin,model_accuracy,sample_count,random_baseline_accuracy
0,0.0-0.1,NaN,0,0.5
1,0.1-0.2,1.000000,12,0.5
2,0.2-0.3,0.800000,30,0.5
3,0.3-0.4,0.838710,62,0.5
4,0.4-0.5,NaN,0,0.5
5,0.5-0.6,NaN,0,0.5
6,0.6-0.7,0.700000,40,0.5
7,0.7-0.8,0.789474,38,0.5
8,0.8-0.9,0.882353,17,0.5
9,0.9-1.0,1.000000,1,0.5
